# 🚀 axionax DeAI Worker - Vertex AI Quickstart

**Platform**: Google Cloud Vertex AI Workbench  
**Purpose**: Test Worker Node and DeAI Training  
**Date**: 2025-11-25

---

## 📋 Notebook Overview

This notebook will help you:
1. ✅ Check GPU and CUDA
2. ✅ Test PyTorch
3. ✅ Run Simple Training (MNIST)
4. ✅ Monitor GPU usage
5. ✅ Save and load models

## 🔧 Part 1: Setup & Verification

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
from datetime import datetime
import json
import matplotlib.pyplot as plt
import numpy as np

print("✅ Libraries imported successfully")

In [ ]:
# Verify PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device("cuda")
else:
    print("⚠️ CUDA not available, using CPU")
    device = torch.device("cpu")

print(f"\n🔧 Using device: {device}")

## 📦 Part 2: Load Dataset

In [ ]:
# Data transformation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST dataset
print("📦 Loading MNIST dataset...")
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"✅ Training samples: {len(train_dataset):,}")
print(f"✅ Test samples: {len(test_dataset):,}")
print(f"✅ Batch size: {batch_size}")

In [ ]:
# Visualize some samples
examples = iter(train_loader)
images, labels = next(examples)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis('off')
plt.suptitle("Sample MNIST Images")
plt.tight_layout()
plt.show()

print(f"Image shape: {images[0].shape}")

## 🏗️ Part 3: Define Model

In [ ]:
# Simple CNN model
class SimpleCNN(nn.Module):
    """Simple CNN for MNIST classification"""
    
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Initialize model
model = SimpleCNN().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️  Model: SimpleCNN")
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 🎓 Part 4: Training

In [ ]:
# Training configuration
epochs = 5
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"⚙️  Training Configuration:")
print(f"  Epochs: {epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Optimizer: Adam")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Device: {device}")

In [ ]:
# Training function
def train_epoch(model, train_loader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)}, '
                  f'Loss: {loss.item():.4f}, '
                  f'Acc: {100.*correct/total:.2f}%')
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

# Test function
def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    avg_loss = test_loss / len(test_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [ ]:
# Training loop
print("\n🎓 Starting training...\n")
training_start = time.time()

history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
    'epoch_time': []
}

for epoch in range(epochs):
    epoch_start = time.time()
    print(f"📚 Epoch {epoch + 1}/{epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, epoch)
    
    # Test
    test_loss, test_acc = test(model, test_loader, criterion)
    
    epoch_time = time.time() - epoch_start
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['epoch_time'].append(epoch_time)
    
    # Summary
    print(f"\n  📊 Epoch {epoch + 1} Summary:")
    print(f"    Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"    Test Loss:  {test_loss:.4f}, Test Acc:  {test_acc:.2f}%")
    print(f"    Time: {epoch_time:.2f}s")
    
    if torch.cuda.is_available():
        print(f"    GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print()

training_time = time.time() - training_start

print("="*60)
print("✅ Training Complete!")
print("="*60)
print(f"⏱️  Total training time: {training_time:.2f}s")
print(f"📈 Final test accuracy: {history['test_acc'][-1]:.2f}%")
print()

## 📊 Part 5: Visualize Results

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['test_loss'], label='Test Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy', marker='o')
ax2.plot(history['test_acc'], label='Test Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Training history plot saved as 'training_history.png'")

## 💾 Part 6: Save Model & Results

In [ ]:
# Save model
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_file = f'model_{timestamp}.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epochs,
    'history': history,
    'final_accuracy': history['test_acc'][-1],
    'config': {
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'epochs': epochs,
        'model': 'SimpleCNN',
        'dataset': 'MNIST'
    }
}, model_file)

print(f"🎯 Model saved to: {model_file}")

# Save results JSON
results_file = f'training_results_{timestamp}.json'
with open(results_file, 'w') as f:
    json.dump({
        'job_id': f'deai_training_{timestamp}',
        'config': {
            'task_type': 'image_classification',
            'model': 'SimpleCNN',
            'dataset': 'MNIST',
            'batch_size': batch_size,
            'epochs': epochs,
            'learning_rate': learning_rate,
            'device': str(device)
        },
        'history': history,
        'total_time_seconds': training_time,
        'final_accuracy': history['test_acc'][-1],
        'gpu_info': {
            'name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'memory_gb': torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else None
        }
    }, f, indent=2)

print(f"💾 Results saved to: {results_file}")

## 🔍 Part 7: Test Predictions

In [ ]:
# Get some test samples
test_examples = iter(test_loader)
images, labels = next(test_examples)
images, labels = images.to(device), labels.to(device)

# Make predictions
model.eval()
with torch.no_grad():
    outputs = model(images)
    _, predictions = torch.max(outputs, 1)

# Visualize predictions
images_cpu = images.cpu()
labels_cpu = labels.cpu()
predictions_cpu = predictions.cpu()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images_cpu[i].squeeze(), cmap='gray')
    color = 'green' if predictions_cpu[i] == labels_cpu[i] else 'red'
    ax.set_title(f"True: {labels_cpu[i]}\nPred: {predictions_cpu[i]}", color=color)
    ax.axis('off')
plt.suptitle("Model Predictions (Green=Correct, Red=Wrong)")
plt.tight_layout()
plt.show()

# Calculate accuracy for this batch
correct = (predictions_cpu == labels_cpu).sum().item()
total = labels_cpu.size(0)
print(f"Batch accuracy: {100 * correct / total:.2f}% ({correct}/{total})")

## 🎮 Part 8: GPU Monitoring

In [ ]:
# GPU stats
if torch.cuda.is_available():
    print("🎮 GPU Statistics:")
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Memory Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"  Max Memory Allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    print(f"  Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # nvidia-smi
    print("\n📊 nvidia-smi output:")
    !nvidia-smi
else:
    print("⚠️ CUDA not available")

## ✅ Summary

This notebook tested:
1. ✅ GPU and CUDA setup
2. ✅ PyTorch training on GPU
3. ✅ MNIST classification with ~98-99% accuracy
4. ✅ Save and load models
5. ✅ Visualize results

### Next Steps:
- ✨ Test other models (ResNet, Transformers)
- ✨ Use larger datasets
- ✨ Distributed training
- ✨ Connect with axionax DeAI network